### 1. Basic Tasks

In [0]:
-- 1. Create a table, make 3 changes to it (insert, update, insert), and use DESCRIBE HISTORY to review the resulting versions.
create table if not exists cyntexa_dev.bronze.products (product_id int, product_name string, product_category string, price double)

In [0]:
insert into table cyntexa_dev.bronze.products values (101, 'Laptop', 'Electronics', 75000.00),
    (102, 'Office Chair', 'Furniture', 8500.00),
    (103, 'Wireless Mouse', 'Electronics', 1200.00),
    (104, 'Notebook', 'Stationery', 150.00),
    (105, 'Desk Lamp', 'Home & Office', 1800.00);

In [0]:
update cyntexa_dev.bronze.products 
set price = 130.00
where product_id = 104;

In [0]:
insert into cyntexa_dev.bronze.products values (106, 'Tablet', 'Electronics', 35000.00)

In [0]:
desc history cyntexa_dev.bronze.products

In [0]:
-- 2. Use COPY INTO to incrementally load 2 batches of files into a bronze table, confirming COPY INTO doesn't reprocess the first batch.

In [0]:
create table if not exists cyntexa_dev.bronze.sales_raw

In [0]:
copy into cyntexa_dev.bronze.sales_raw
from "/Volumes/cyntexa_dev/bronze/raw/sales/"
fileformat = csv
format_options ('header' = 'true')
copy_options ('mergeSchema' = 'true')


In [0]:
select count(*) from cyntexa_dev.bronze.sales_raw

Number of rows inserted : 78

Total numbr of records after first run :  78

In [0]:
copy into cyntexa_dev.bronze.sales_raw
from "/Volumes/cyntexa_dev/bronze/raw/sales/"
fileformat = csv
format_options ('header' = 'true')
copy_options ('mergeSchema' = 'true')


In [0]:
select count(*) from cyntexa_dev.bronze.sales_raw

Number of rows inserted : 77

Total number of records after second run :  155

This confirms COPY INTO did not reload files form first batch 

In [0]:
-- 3. Query an old version of the table with both VERSION AS OF and TIMESTAMP AS OF.

In [0]:
desc history cyntexa_dev.bronze.products

In [0]:
select * from cyntexa_dev.bronze.products version as of 2

In [0]:
select * from cyntexa_dev.bronze.products timestamp as of '2026-08-26T08:29:43.000+00:00' --version 3

### 2. Intermediate Tasks

In [0]:
-- 4. Evolve the table's schema two ways: append a new column using mergeSchema, then change an existing column's type using overwriteSchema; document the difference in what each requires.

#### Merge schema

In [0]:
%python
df = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers/customers1.csv', header = True, inferSchema = True)
df.display()

In [0]:
%python
df.write.mode("overwrite").saveAsTable("cyntexa_dev.sales.sales_schema_evolution")

In [0]:
%python
df2 = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers/customers2.csv', header = True, inferSchema = True)
df2.display()

In [0]:
%python
df2.write.mode("append").option('mergeSchema', True).saveAsTable("cyntexa_dev.sales.sales_schema_evolution")

In [0]:
select * from cyntexa_dev.sales.sales_schema_evolution

#### Overwrite schema

In [0]:
%python
df3 = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers/customers3.csv', header=True, inferSchema=True,sep='|')
df3.display()

In [0]:
%python
from pyspark.sql.functions import *
df3 = df3.withColumn("customer_id", concat(lit("C"), col("customer_id")))
df3.display()

In [0]:
%python
df3.printSchema()

Overwriting schmema by changing customer_id data type to string

In [0]:
%python
df3.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("cyntexa_dev.sales.sales_schema_overwrite")

In [0]:
select * from cyntexa_dev.sales.sales_schema_overwrite